# NB_bishop_ch02_figures

**Bishop Chapter 2 — Key Figures: Sum & Product Rules, Bayes' Theorem, Gaussian Distribution, Likelihood, Prior & Posterior**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_02/NB_bishop_ch02_figures.ipynb)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from scipy import stats
import os

# ── Global styling ──────────────────────────────────────────────────
COLORS = {
    'blue':  '#1A3A6E',
    'red':   '#CD0000',
    'green': '#2E7D32',
    'amber': '#B5853F',
    'purple': '#8E44AD',
}

matplotlib.rcParams['figure.facecolor']   = 'none'
matplotlib.rcParams['axes.facecolor']     = 'none'
matplotlib.rcParams['savefig.facecolor']  = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid']          = False
matplotlib.rcParams['legend.framealpha']  = 0.0
matplotlib.rcParams['font.size']          = 11

CHART_DIR = '../../charts'
os.makedirs(CHART_DIR, exist_ok=True)
print(f'Charts will be saved to: {os.path.abspath(CHART_DIR)}')


def save_fig(fig, name, chart_dir=CHART_DIR):
    """Save figure as PDF + PNG with transparent background."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=300)
    print(f'Saved: {name}.pdf / .png')

## Figure 2.1 — Joint and Marginal Distributions

Joint distribution $p(X, Y)$ shown as a heatmap for two discrete random variables, with marginal distributions $p(X)$ and $p(Y)$ displayed as bar charts along the axes. Illustrates how marginals are obtained by summing over the other variable (the **sum rule**).

In [ ]:
np.random.seed(42)

# Define a joint distribution over two discrete variables
# X has 8 categories, Y has 5 categories
n_x, n_y = 8, 5
# Create a structured joint distribution (not uniform)
joint = np.array([
    [0.02, 0.04, 0.03, 0.01, 0.01],
    [0.03, 0.06, 0.05, 0.02, 0.01],
    [0.01, 0.03, 0.08, 0.04, 0.02],
    [0.01, 0.02, 0.06, 0.06, 0.03],
    [0.00, 0.01, 0.03, 0.05, 0.04],
    [0.01, 0.01, 0.02, 0.03, 0.03],
    [0.00, 0.01, 0.01, 0.02, 0.02],
    [0.00, 0.00, 0.01, 0.01, 0.01],
])
joint = joint / joint.sum()  # normalize

# Marginals via the sum rule
p_x = joint.sum(axis=1)  # sum over Y
p_y = joint.sum(axis=0)  # sum over X

# ── Plot ──
fig = plt.figure(figsize=(8, 7))

# Layout: heatmap in center, marginal_x on right, marginal_y on top
gs = fig.add_gridspec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
                      hspace=0.05, wspace=0.05)

ax_joint = fig.add_subplot(gs[1, 0])
ax_marg_y = fig.add_subplot(gs[0, 0], sharex=ax_joint)
ax_marg_x = fig.add_subplot(gs[1, 1], sharey=ax_joint)
ax_empty = fig.add_subplot(gs[0, 1])
ax_empty.axis('off')

# Joint heatmap
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('custom', ['#FFFFFF', COLORS['blue']])
im = ax_joint.imshow(joint, cmap=cmap, aspect='auto', origin='lower')
ax_joint.set_xlabel('$Y$', fontsize=13)
ax_joint.set_ylabel('$X$', fontsize=13)
ax_joint.set_xticks(range(n_y))
ax_joint.set_yticks(range(n_x))
ax_joint.set_xticklabels([f'$y_{{{i+1}}}$' for i in range(n_y)])
ax_joint.set_yticklabels([f'$x_{{{i+1}}}$' for i in range(n_x)])

# Annotate cells with probabilities
for i in range(n_x):
    for j in range(n_y):
        val = joint[i, j]
        color = 'white' if val > 0.04 else COLORS['blue']
        ax_joint.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

# Marginal p(Y) — top bar chart
ax_marg_y.bar(range(n_y), p_y, color=COLORS['red'], alpha=0.85, edgecolor='white')
ax_marg_y.set_ylabel('$p(Y)$', fontsize=11)
ax_marg_y.set_xlim(-0.5, n_y - 0.5)
plt.setp(ax_marg_y.get_xticklabels(), visible=False)
ax_marg_y.tick_params(axis='x', length=0)

# Marginal p(X) — right bar chart
ax_marg_x.barh(range(n_x), p_x, color=COLORS['green'], alpha=0.85, edgecolor='white')
ax_marg_x.set_xlabel('$p(X)$', fontsize=11)
ax_marg_x.set_ylim(-0.5, n_x - 0.5)
plt.setp(ax_marg_x.get_yticklabels(), visible=False)
ax_marg_x.tick_params(axis='y', length=0)

fig.suptitle('Joint Distribution $p(X, Y)$ with Marginals', fontsize=13, y=0.95)

save_fig(fig, 'fig2_1_joint_marginal')
plt.show()

## Figure 2.2 — Sum Rule Illustration

The sum rule: $p(X) = \sum_Y p(X, Y)$. We highlight one row of the joint distribution table and show how summing across $Y$ yields $p(X = x_3)$.

In [ ]:
# Sum rule: highlight row x_3 and show summing across Y
highlight_row = 2  # x_3 (0-indexed)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1: Full joint distribution with highlighted row
ax = axes[0]
ax.imshow(joint, cmap=cmap, aspect='auto', origin='lower', alpha=0.4)
# Highlight the row
row_data = np.zeros_like(joint)
row_data[highlight_row, :] = joint[highlight_row, :]
ax.imshow(np.where(row_data > 0, joint, np.nan), cmap=cmap, aspect='auto', origin='lower', vmin=0, vmax=joint.max())
# Draw rectangle around highlighted row
rect = plt.Rectangle((-0.5, highlight_row - 0.5), n_y, 1, linewidth=2.5,
                      edgecolor=COLORS['red'], facecolor='none')
ax.add_patch(rect)
ax.set_xticks(range(n_y))
ax.set_yticks(range(n_x))
ax.set_xticklabels([f'$y_{{{i+1}}}$' for i in range(n_y)])
ax.set_yticklabels([f'$x_{{{i+1}}}$' for i in range(n_x)])
ax.set_xlabel('$Y$')
ax.set_ylabel('$X$')
ax.set_title('$p(X, Y)$  — row $x_3$ highlighted', fontsize=11)
for i in range(n_x):
    for j in range(n_y):
        alpha = 1.0 if i == highlight_row else 0.3
        ax.text(j, i, f'{joint[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color=COLORS['blue'], alpha=alpha)

# Panel 2: The highlighted row values
ax = axes[1]
row_vals = joint[highlight_row, :]
bars = ax.bar(range(n_y), row_vals, color=COLORS['red'], alpha=0.8, edgecolor='white')
for i, v in enumerate(row_vals):
    ax.text(i, v + 0.002, f'{v:.3f}', ha='center', va='bottom', fontsize=9, color=COLORS['red'])
ax.set_xticks(range(n_y))
ax.set_xticklabels([f'$p(x_3, y_{{{i+1}}})$' for i in range(n_y)], fontsize=8)
ax.set_ylabel('Probability')
ax.set_title(r'$p(X\!=\!x_3,\, Y)$ for each $Y$', fontsize=11)
ax.set_ylim(0, row_vals.max() * 1.4)

# Panel 3: Result — the marginal value
ax = axes[2]
marginal_val = row_vals.sum()
ax.bar([0], [marginal_val], color=COLORS['green'], alpha=0.85, width=0.5, edgecolor='white')
ax.text(0, marginal_val + 0.005, f'{marginal_val:.3f}', ha='center', va='bottom',
        fontsize=12, color=COLORS['green'], fontweight='bold')
ax.set_xticks([0])
ax.set_xticklabels([r'$p(X\!=\!x_3)$'], fontsize=11)
ax.set_ylabel('Probability')
ax.set_title(r'$p(x_3) = \sum_Y p(x_3, Y)$', fontsize=11)
ax.set_ylim(0, marginal_val * 1.5)
ax.set_xlim(-0.5, 0.5)

# Add arrows between panels
fig.subplots_adjust(wspace=0.35)
for i in range(2):
    fig.text(0.365 + i * 0.32, 0.5, r'$\Longrightarrow$', fontsize=22,
             ha='center', va='center', color=COLORS['amber'])

save_fig(fig, 'fig2_2_sum_rule')
plt.show()

## Figure 2.3 — Product Rule and Bayes' Theorem

Illustrates the product rule $p(X, Y) = p(Y|X)\,p(X)$ and Bayes' theorem $p(Y|X) = \frac{p(X|Y)\,p(Y)}{p(X)}$ using continuous densities.

In [ ]:
# Product rule / Bayes' theorem diagram
# Use a concrete example: two Gaussians
# p(X) ~ N(2, 1), p(Y|X=x) ~ N(x, 0.5)  =>  p(X|Y=y) via Bayes

x_range = np.linspace(-2, 6, 500)

# Prior p(X)
prior_mu, prior_sigma = 2.0, 1.0
p_x = stats.norm.pdf(x_range, prior_mu, prior_sigma)

# Likelihood p(Y=3 | X=x) — for observed y=3
y_obs = 3.0
likelihood_sigma = 0.7
likelihood = stats.norm.pdf(y_obs, x_range, likelihood_sigma)

# Posterior p(X | Y=3) via Bayes (for Gaussian conjugate: known formula)
post_var = 1.0 / (1.0 / prior_sigma**2 + 1.0 / likelihood_sigma**2)
post_mu = post_var * (prior_mu / prior_sigma**2 + y_obs / likelihood_sigma**2)
post_sigma = np.sqrt(post_var)
posterior = stats.norm.pdf(x_range, post_mu, post_sigma)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

# Panel 1: p(X) — prior
ax = axes[0]
ax.fill_between(x_range, p_x, alpha=0.3, color=COLORS['blue'])
ax.plot(x_range, p_x, color=COLORS['blue'], lw=2.5)
ax.set_title('Prior $p(X)$', fontsize=12)
ax.set_xlabel('$X$')
ax.set_ylabel('Density')
ax.set_xlim(-1, 5.5)

# Panel 2: p(Y=3|X) — likelihood
ax = axes[1]
ax.fill_between(x_range, likelihood, alpha=0.3, color=COLORS['red'])
ax.plot(x_range, likelihood, color=COLORS['red'], lw=2.5)
ax.set_title('Likelihood $p(Y\\!=\\!3 \\mid X)$', fontsize=12)
ax.set_xlabel('$X$')
ax.set_xlim(-1, 5.5)

# Panel 3: p(X|Y=3) — posterior
ax = axes[2]
ax.fill_between(x_range, posterior, alpha=0.3, color=COLORS['green'])
ax.plot(x_range, posterior, color=COLORS['green'], lw=2.5)
ax.set_title('Posterior $p(X \\mid Y\\!=\\!3)$', fontsize=12)
ax.set_xlabel('$X$')
ax.set_xlim(-1, 5.5)

# Multiplication and division signs between panels
fig.subplots_adjust(wspace=0.3)
fig.text(0.365, 0.5, r'$\times$', fontsize=26, ha='center', va='center',
         color=COLORS['amber'], fontweight='bold')
fig.text(0.655, 0.5, r'$\propto$', fontsize=22, ha='center', va='center',
         color=COLORS['amber'], fontweight='bold')

save_fig(fig, 'fig2_3_product_rule_bayes')
plt.show()

## Figure 2.4 — Prior, Likelihood, and Posterior

Three properly normalized curves overlaid on the same axis: prior $p(\mu)$, likelihood $p(\mathcal{D}|\mu)$, and posterior $p(\mu|\mathcal{D})$. Demonstrates how the posterior is a compromise between prior belief and data evidence.

In [ ]:
# Prior x Likelihood = Posterior — overlaid on single axis
mu_range = np.linspace(-2, 7, 500)

# Prior: broad Gaussian
prior_mu_val, prior_sigma_val = 1.5, 1.5
prior_curve = stats.norm.pdf(mu_range, prior_mu_val, prior_sigma_val)

# Likelihood: narrower, centered on observed data mean
# Simulate N=5 observations from N(4, 1)
np.random.seed(7)
data = np.random.normal(4.0, 1.0, 5)
data_mean = data.mean()
# Likelihood for the mean (known variance sigma=1): L(mu) = N(data_mean, sigma/sqrt(N))
lik_sigma = 1.0 / np.sqrt(len(data))
lik_curve = stats.norm.pdf(mu_range, data_mean, lik_sigma)
# Normalize likelihood to have same peak scale for visual comparison
lik_curve_scaled = lik_curve / lik_curve.max() * max(prior_curve.max(), 0.4)

# Posterior (conjugate Gaussian)
post_prec = 1.0 / prior_sigma_val**2 + len(data) / 1.0**2
post_var_val = 1.0 / post_prec
post_mu_val = post_var_val * (prior_mu_val / prior_sigma_val**2 + data.sum() / 1.0**2)
post_sigma_val = np.sqrt(post_var_val)
post_curve = stats.norm.pdf(mu_range, post_mu_val, post_sigma_val)

fig, ax = plt.subplots(figsize=(8, 5))

ax.fill_between(mu_range, prior_curve, alpha=0.15, color=COLORS['blue'])
ax.plot(mu_range, prior_curve, color=COLORS['blue'], lw=2.5, label=r'Prior $p(\mu)$')

ax.fill_between(mu_range, lik_curve_scaled, alpha=0.15, color=COLORS['red'])
ax.plot(mu_range, lik_curve_scaled, color=COLORS['red'], lw=2.5, ls='--',
        label=r'Likelihood $p(\mathcal{D}\mid\mu)$ (scaled)')

ax.fill_between(mu_range, post_curve, alpha=0.15, color=COLORS['green'])
ax.plot(mu_range, post_curve, color=COLORS['green'], lw=2.5,
        label=r'Posterior $p(\mu\mid\mathcal{D})$')

# Mark the means
for mu_val, color, marker in [(prior_mu_val, COLORS['blue'], 'v'),
                               (data_mean, COLORS['red'], '^'),
                               (post_mu_val, COLORS['green'], 'o')]:
    ax.axvline(mu_val, color=color, ls=':', lw=1, alpha=0.6)

# Data points on x-axis
ax.scatter(data, np.zeros_like(data) - 0.01, marker='|', s=200, color=COLORS['red'],
           zorder=5, linewidths=2, label=f'Data ($N={len(data)}$)')

ax.set_xlabel(r'$\mu$', fontsize=13)
ax.set_ylabel('Density', fontsize=12)
ax.set_xlim(-1.5, 7)
ax.set_ylim(-0.03, None)
ax.legend(fontsize=10)

save_fig(fig, 'fig2_4_prior_posterior')
plt.show()

## Figure 2.5 — Gaussian Distribution with Different Parameters

2x2 panel showing $\mathcal{N}(\mu, \sigma^2)$ for different combinations of mean $\mu$ and standard deviation $\sigma$, illustrating the effect of each parameter on the shape of the distribution.

In [ ]:
x_gauss = np.linspace(-6, 8, 500)

params = [
    (0, 1.0, '(a)'),   # standard normal
    (0, 2.0, '(b)'),   # wider
    (2, 1.0, '(c)'),   # shifted right
    (2, 0.5, '(d)'),   # shifted and narrow
]

colors_list = [COLORS['blue'], COLORS['red'], COLORS['green'], COLORS['purple']]

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)

for ax, (mu, sigma, label), color in zip(axes.flat, params, colors_list):
    y = stats.norm.pdf(x_gauss, mu, sigma)
    ax.fill_between(x_gauss, y, alpha=0.2, color=color)
    ax.plot(x_gauss, y, color=color, lw=2.5)
    ax.axvline(mu, color=color, ls=':', lw=1, alpha=0.5)
    ax.set_title(f'{label}  $\\mu = {mu},\\; \\sigma = {sigma}$', fontsize=11)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$p(x)$')
    ax.set_xlim(-5, 7)

fig.suptitle('Gaussian $\\mathcal{N}(\\mu, \\sigma^2)$ with Different Parameters', fontsize=13, y=1.0)
fig.tight_layout()
save_fig(fig, 'fig2_5_gaussian_params')
plt.show()

## Figure 2.6 — Likelihood Function

Given observed data points $\{x_1, \ldots, x_N\}$ drawn from $\mathcal{N}(\mu, 1)$, plot the likelihood $L(\mu) = \prod_n p(x_n \mid \mu)$ and the log-likelihood as functions of $\mu$. The MLE $\hat{\mu}_{\text{ML}} = \bar{x}$ is marked.

In [ ]:
# Likelihood function for Gaussian mean
np.random.seed(12)
observed = np.random.normal(2.5, 1.0, 7)  # N=7 observations
sigma_known = 1.0

mu_grid = np.linspace(-1, 6, 500)

# Log-likelihood: sum of log N(x_n | mu, sigma^2)
log_lik = np.array([
    np.sum(stats.norm.logpdf(observed, loc=mu, scale=sigma_known))
    for mu in mu_grid
])
# Likelihood (normalized so max = 1 for visibility)
lik = np.exp(log_lik - log_lik.max())

mu_mle = observed.mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel 1: Likelihood
ax = axes[0]
ax.fill_between(mu_grid, lik, alpha=0.2, color=COLORS['blue'])
ax.plot(mu_grid, lik, color=COLORS['blue'], lw=2.5)
ax.axvline(mu_mle, color=COLORS['red'], ls='--', lw=1.5, label=f'$\\hat{{\\mu}}_{{\\mathrm{{ML}}}} = {mu_mle:.2f}$')
ax.scatter(observed, np.zeros_like(observed) - 0.02, marker='|', s=200,
           color=COLORS['red'], linewidths=2, zorder=5, label=f'Data ($N={len(observed)}$)')
ax.set_xlabel(r'$\mu$', fontsize=13)
ax.set_ylabel(r'$L(\mu)$ (normalized)', fontsize=12)
ax.set_title('Likelihood Function', fontsize=12)
ax.set_ylim(-0.05, 1.1)
ax.legend(fontsize=10)

# Panel 2: Log-likelihood
ax = axes[1]
ax.plot(mu_grid, log_lik, color=COLORS['green'], lw=2.5)
ax.axvline(mu_mle, color=COLORS['red'], ls='--', lw=1.5, label=f'$\\hat{{\\mu}}_{{\\mathrm{{ML}}}} = {mu_mle:.2f}$')
ax.set_xlabel(r'$\mu$', fontsize=13)
ax.set_ylabel(r'$\ln L(\mu)$', fontsize=12)
ax.set_title('Log-Likelihood Function', fontsize=12)
ax.legend(fontsize=10)

fig.tight_layout()
save_fig(fig, 'fig2_6_likelihood')
plt.show()

## Figure 2.7 — MLE Variance Bias

The MLE variance estimator $\hat{\sigma}^2_{\text{ML}} = \frac{1}{N}\sum(x_n - \bar{x})^2$ is biased: $\mathbb{E}[\hat{\sigma}^2_{\text{ML}}] = \frac{N-1}{N}\sigma^2$. This plot shows the expected ratio $\mathbb{E}[\hat{\sigma}^2_{\text{ML}}]/\sigma^2$ vs $N$, compared to the unbiased estimator $s^2 = \frac{1}{N-1}\sum(x_n - \bar{x})^2$. A Monte Carlo simulation confirms the theoretical bias.

In [ ]:
# MLE variance bias demonstration
np.random.seed(0)
true_sigma2 = 1.0
true_mu = 0.0
n_sims = 10000
N_values = np.arange(2, 51)

# Theoretical bias ratio
theory_ratio = (N_values - 1) / N_values

# Monte Carlo simulation
mc_mle_ratio = []
mc_unbiased_ratio = []

for N in N_values:
    samples = np.random.normal(true_mu, np.sqrt(true_sigma2), size=(n_sims, N))
    means = samples.mean(axis=1, keepdims=True)
    # MLE variance (divide by N)
    mle_var = ((samples - means)**2).sum(axis=1) / N
    # Unbiased variance (divide by N-1)
    unbiased_var = ((samples - means)**2).sum(axis=1) / (N - 1)
    mc_mle_ratio.append(mle_var.mean() / true_sigma2)
    mc_unbiased_ratio.append(unbiased_var.mean() / true_sigma2)

fig, ax = plt.subplots(figsize=(8, 5))

ax.axhline(1.0, color='gray', ls='-', lw=1, alpha=0.5, label=r'True $\sigma^2$')

ax.plot(N_values, theory_ratio, color=COLORS['blue'], lw=2.5, ls='-',
        label=r'$\mathbb{E}[\hat{\sigma}^2_{\mathrm{ML}}]/\sigma^2 = (N\!-\!1)/N$')
ax.scatter(N_values[::3], np.array(mc_mle_ratio)[::3], color=COLORS['red'], s=30, zorder=5,
           label=r'Monte Carlo $\hat{\sigma}^2_{\mathrm{ML}}$')
ax.scatter(N_values[::3], np.array(mc_unbiased_ratio)[::3], color=COLORS['green'], s=30,
           marker='s', zorder=5, label=r'Monte Carlo $s^2$ (unbiased)')

ax.set_xlabel('$N$ (sample size)', fontsize=12)
ax.set_ylabel(r'$\mathbb{E}[\hat{\sigma}^2] \,/\, \sigma^2$', fontsize=12)
ax.set_title('MLE Variance Estimator Bias', fontsize=13)
ax.set_xlim(1, 51)
ax.set_ylim(0.3, 1.15)
ax.legend(fontsize=10)

save_fig(fig, 'fig2_7_mle_bias')
plt.show()

## Figure 2.8 — Bayesian Polynomial Curve Fitting

Bayesian approach to polynomial regression: instead of a single MLE fit, we compute the posterior over weights and show the **predictive distribution** with uncertainty bands ($\pm 1\sigma$). As data increases, uncertainty shrinks.

In [ ]:
# Bayesian polynomial curve fitting
# Using a degree-9 polynomial with Gaussian prior on weights

np.random.seed(42)
N_train = 10
x_train = np.linspace(0, 1, N_train)
t_train = np.sin(2 * np.pi * x_train) + np.random.randn(N_train) * 0.3
x_dense = np.linspace(0, 1, 200)

M = 9            # polynomial degree
alpha = 5e-3     # prior precision (small = broad prior)
beta = 25.0      # noise precision (1/sigma_noise^2)

def design_matrix(x, degree):
    """Vandermonde design matrix (increasing powers)."""
    return np.vander(x, degree + 1, increasing=True)

Phi_train = design_matrix(x_train, M)
Phi_dense = design_matrix(x_dense, M)

# Posterior over weights: p(w|t) = N(w|m_N, S_N)
S_N_inv = alpha * np.eye(M + 1) + beta * Phi_train.T @ Phi_train
S_N = np.linalg.inv(S_N_inv)
m_N = beta * S_N @ Phi_train.T @ t_train

# Predictive distribution at each test point
y_mean = Phi_dense @ m_N
y_var = np.array([1.0 / beta + phi @ S_N @ phi for phi in Phi_dense])
y_std = np.sqrt(y_var)

# Plot for different amounts of training data
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
n_points_list = [1, 2, 4, N_train]

for ax, n_pts in zip(axes.flat, n_points_list):
    # Recompute posterior with first n_pts data points
    Phi_sub = design_matrix(x_train[:n_pts], M)
    S_inv = alpha * np.eye(M + 1) + beta * Phi_sub.T @ Phi_sub
    S = np.linalg.inv(S_inv)
    m = beta * S @ Phi_sub.T @ t_train[:n_pts]

    y_m = Phi_dense @ m
    y_v = np.array([1.0 / beta + phi @ S @ phi for phi in Phi_dense])
    y_s = np.sqrt(y_v)

    ax.plot(x_dense, np.sin(2 * np.pi * x_dense), color=COLORS['green'], lw=1.5, alpha=0.5)
    ax.fill_between(x_dense, y_m - y_s, y_m + y_s, alpha=0.25, color=COLORS['red'])
    ax.plot(x_dense, y_m, color=COLORS['red'], lw=2)
    ax.scatter(x_train[:n_pts], t_train[:n_pts], color=COLORS['blue'], s=50,
               zorder=5, edgecolors='white', linewidths=0.5)
    ax.set_title(f'$N = {n_pts}$', fontsize=11)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.8, 1.8)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$t$')

fig.suptitle('Bayesian Polynomial Regression — Predictive Distribution', fontsize=13, y=1.0)
fig.tight_layout()
save_fig(fig, 'fig2_8_bayesian_curve_fitting')
plt.show()

## Figure 2.9 — Posterior Predictive Distribution

Full Bayesian predictive distribution for the polynomial model using all $N=10$ data points. Shows the predictive mean, $\pm 1\sigma$ and $\pm 2\sigma$ uncertainty bands, and sampled functions from the posterior.

In [ ]:
# Full predictive distribution with sampled functions from posterior
fig, ax = plt.subplots(figsize=(8, 5))

# True function
ax.plot(x_dense, np.sin(2 * np.pi * x_dense), color=COLORS['green'], lw=1.5, alpha=0.5,
        label=r'$\sin(2\pi x)$')

# Uncertainty bands (using the full posterior computed above with all N_train points)
ax.fill_between(x_dense, y_mean - 2*y_std, y_mean + 2*y_std, alpha=0.1, color=COLORS['red'],
                label=r'$\pm 2\sigma$')
ax.fill_between(x_dense, y_mean - y_std, y_mean + y_std, alpha=0.25, color=COLORS['red'],
                label=r'$\pm 1\sigma$')

# Sample functions from the posterior
np.random.seed(3)
n_samples = 5
w_samples = np.random.multivariate_normal(m_N, S_N, size=n_samples)
for i, w_s in enumerate(w_samples):
    y_s = Phi_dense @ w_s
    ax.plot(x_dense, y_s, color=COLORS['purple'], lw=0.8, alpha=0.6,
            label='Posterior samples' if i == 0 else None)

# Predictive mean
ax.plot(x_dense, y_mean, color=COLORS['red'], lw=2.5, label='Predictive mean')

# Training data
ax.scatter(x_train, t_train, color=COLORS['blue'], s=60, zorder=5,
           edgecolors='white', linewidths=0.5, label='Training data')

ax.set_xlabel('$x$', fontsize=13)
ax.set_ylabel('$t$', fontsize=12)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-1.8, 1.8)
ax.legend(fontsize=9, ncol=2)

save_fig(fig, 'fig2_9_posterior_predictive')
plt.show()

## Figure 2.10 — Entropy of Bernoulli Distribution

The entropy $H(p) = -p\ln p - (1-p)\ln(1-p)$ of a Bernoulli random variable as a function of the success probability $p$. Maximum entropy occurs at $p = 0.5$ (maximum uncertainty).

In [ ]:
# Entropy of Bernoulli distribution
p = np.linspace(1e-6, 1 - 1e-6, 500)

# Using natural log (nats)
H_nats = -p * np.log(p) - (1 - p) * np.log(1 - p)
# Using log base 2 (bits)
H_bits = -p * np.log2(p) - (1 - p) * np.log2(1 - p)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel (a): Entropy in nats
ax = axes[0]
ax.plot(p, H_nats, color=COLORS['blue'], lw=2.5)
ax.fill_between(p, H_nats, alpha=0.15, color=COLORS['blue'])
ax.axvline(0.5, color=COLORS['red'], ls='--', lw=1.5, alpha=0.7)
ax.scatter([0.5], [np.log(2)], color=COLORS['red'], s=60, zorder=5)
ax.annotate(f'$\\ln 2 \\approx {np.log(2):.3f}$', xy=(0.5, np.log(2)),
            xytext=(0.65, 0.55), fontsize=10, color=COLORS['red'],
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1.2))
ax.set_xlabel('$p$', fontsize=13)
ax.set_ylabel('$H(p)$ (nats)', fontsize=12)
ax.set_title('(a)  Entropy in nats', fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 0.8)

# Panel (b): Entropy in bits
ax = axes[1]
ax.plot(p, H_bits, color=COLORS['green'], lw=2.5)
ax.fill_between(p, H_bits, alpha=0.15, color=COLORS['green'])
ax.axvline(0.5, color=COLORS['red'], ls='--', lw=1.5, alpha=0.7)
ax.scatter([0.5], [1.0], color=COLORS['red'], s=60, zorder=5)
ax.annotate('$1$ bit', xy=(0.5, 1.0), xytext=(0.65, 0.75), fontsize=10,
            color=COLORS['red'],
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1.2))
ax.set_xlabel('$p$', fontsize=13)
ax.set_ylabel('$H(p)$ (bits)', fontsize=12)
ax.set_title('(b)  Entropy in bits', fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.1)

fig.tight_layout()
save_fig(fig, 'fig2_10_entropy')
plt.show()

## Summary

All figures have been saved to the `charts/` directory as both PDF and PNG (300 dpi, transparent background).

In [ ]:
# List all generated chart files
chart_files = sorted([f for f in os.listdir(CHART_DIR) if f.startswith('fig2_')])
print(f'Generated {len(chart_files)} files in {CHART_DIR}:')
for f in chart_files:
    print(f'  {f}')